In [1]:
from pathlib import Path

In [2]:
import torch
from transformers import set_seed, AutoTokenizer, AutoModelForCausalLM
set_seed(42)
from peft import PeftModel

In [3]:
ckpt_dir = Path("data") / "ch12" / "checkpoint-500"
lora_dir = Path("data") / "ch12" / "sft-lora" / "checkpoint-5800"

In [4]:
tokenizer = AutoTokenizer.from_pretrained(ckpt_dir, use_fast=True)

# model = LlamaForCausalLM.from_pretrained(
base_model = AutoModelForCausalLM.from_pretrained(
    ckpt_dir,
    local_files_only=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto",           # 多卡自动放置
    attn_implementation="sdpa",  # 若已安装 flash-attn, 则使用 flash_attention_2；否则删掉或改 "sdpa"

    #quantization_config=bnb,
)

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

Memory footprint: 115.8 MB


In [5]:
model = PeftModel.from_pretrained(base_model, lora_dir)

#def llm(prompt):
#    inputs = tokenizer.encode(prompt, return_tensors="pt").to("cuda")
#    attention_mask = torch.ones(inputs.shape, device="cuda")
#    outputs = model.generate(inputs, attention_mask=attention_mask, max_new_tokens=3, num_return_sequences=1)
#    response = tokenizer.decode(outputs[0])
#    return response

#if (data_dir/ "adapter_config.json").exists():
#    model = PeftModel.from_pretrained(model, data_dir)
#else:
#    peft_model = get_peft_model(model, lora_config)
#    peft_model.save_pretrained(data_dir)


#print(model.active_adapter)                     # 当前适配器
#model.load_adapter("path/to/adapter2", adapter_name="task2")
#model.set_adapter("task2")                      # 切换到另一个适配器
# model.disable_adapter()                       # 关闭适配器，仅用基座

model.eval()
model.config.use_cache = True

print(f"Memory footprint: {model.get_memory_footprint() / 1e6:.1f} MB")

Memory footprint: 120.0 MB


In [6]:
prompt = "#### User: What color is the sky?" # \n#### Assistant:
inp = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

print(inp)

with torch.inference_mode():
    out = model.generate(
        inp,
        eos_token_id=[tokenizer.eos_token_id],
        pad_token_id=tokenizer.pad_token_id,
        max_new_tokens=128,
        do_sample=True,
        top_p=0.9,
        temperature=0.8,
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))

tensor([[    2, 29619,  2563,  5793,  2100, 12537,  9122,  8488,    39,     3]],
       device='cuda:0')
#### User: What color is the sky?#### User: I am writer!?
#### Assistant: What is the best following number requests? What that includes a specific model for a time?
#### User: I am very popular:

#### Assistant: The basic example "time"


In [12]:
#base = AutoModelForCausalLM.from_pretrained(
#    base_id,
#    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
#    device_map="auto",
#)

#merged = PeftModel.from_pretrained(base, adapter_dir)

merged_dir = Path("data") / "ch12" / "sft-lora" / "merged"
merged = model.merge_and_unload()
merged.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)

('data/ch12/sft-lora/merged/tokenizer_config.json',
 'data/ch12/sft-lora/merged/special_tokens_map.json',
 'data/ch12/sft-lora/merged/tokenizer.json')